# P5-6 · GĐ 6 · Đánh giá và rectified flow

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
def frechet(a, b):                         # a, b: (N, d) đặc trưng của hai tập
    ma, mb = a.mean(0), b.mean(0)
    ca, cb = torch.cov(a.T), torch.cov(b.T)
    tr_sqrt = torch.linalg.eigvals(ca @ cb).real.clamp(min=0).sqrt().sum()   # Tr((Σa Σb)^½)
    return ((ma - mb) ** 2).sum() + ca.trace() + cb.trace() - 2 * tr_sqrt

## Bài tập 6: Flow matching trên điểm 2D {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

def make_spiral(n):
    th = torch.sqrt(torch.rand(n)) * 3 * math.pi
    return torch.stack([th * torch.cos(th), th * torch.sin(th)], 1) / (3 * math.pi) + 0.02 * torch.randn(n, 2)

class VNet(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, t[:, None]], 1))

data, model = make_spiral(10000), VNet()
opt = torch.optim.Adam(model.parameters(), 1e-3)
for step in range(4000):
    x0 = data[torch.randint(0, len(data), (512,))]
    eps, t = torch.randn_like(x0), torch.rand(512)
    xt = (1 - t[:, None]) * x0 + t[:, None] * eps
    loss = F.mse_loss(model(xt, t), eps - x0)
    opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def sample(n=2000, steps=10):
    x, dt = torch.randn(n, 2), 1.0 / steps
    for i in range(steps, 0, -1):
        x = x - dt * model(x, torch.full((n,), i * dt))
    return x

pts = sample()
print("khoảng cách TB tới dữ liệu:", torch.cdist(pts, data[:5000]).min(1).values.mean().item())

khoảng cách TB tới dữ liệu: 0.04033367708325386
